In [1]:
import sys
import pandas as pd

# Add the src folder to Python's path so we can import our custom module
sys.path.append('../src')
from data_prep import load_and_split_data

# Load the time-split data
train_data, test_data, orders = load_and_split_data()

print(f"Train rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")

Loading datasets...
Merging data to create time-based Train/Test splits...
Split complete!
Train rows: 32,434,489
Test rows: 1,384,617


In [2]:
import numpy as np

# 1. Build Baseline 1: Global Popularity
# Recommend the top 10 most frequently bought products overall
top_10_popular = train_data['product_id'].value_counts().head(10).index.tolist()
print(f"Top 10 Popular Product IDs: {top_10_popular}")

# 2. Prepare the Test Data for Evaluation
# Group the test data to see exactly what each user bought in their final order
print("Structuring test data for evaluation...")
actual_purchases = test_data.groupby('user_id')['product_id'].apply(set).to_dict()

# 3. Create the Evaluation Function (Recall@K)
def calculate_recall_at_k(actual_dict, predicted_list, k=10):
    recalls = []
    # Evaluate every user in the test set
    for user_id, true_items in actual_dict.items():
        if not true_items:
            continue
        # Count how many of our top K predictions match what they actually bought
        hits = len(set(predicted_list[:k]) & true_items)
        recalls.append(hits / len(true_items))
    return np.mean(recalls)

# 4. Score Baseline 1
print("Scoring Baseline 1...")
baseline1_score = calculate_recall_at_k(actual_purchases, top_10_popular, k=10)
print(f"Baseline 1 (Popular Items) Recall@10: {baseline1_score:.4f}")

Top 10 Popular Product IDs: [24852, 13176, 21137, 21903, 47209, 47766, 47626, 16797, 26209, 27845]
Structuring test data for evaluation...
Scoring Baseline 1...
Baseline 1 (Popular Items) Recall@10: 0.0698
